In [1]:
def get_weather(city):
    return f"The weather in {city} is sunny."


print(get_weather("Hyderabad"))

The weather in Hyderabad is sunny.


In [2]:
from smolagents import tool

@tool
def get_weather(city: str) -> str:
    """Returns the weather for a given city.

    Args:
        city: The city whose weather should be returned.

    Returns:
        A short weather description.
    """
    return f"The weather in {city} is sunny."


print(get_weather("Hyderabad"))
# the triple quotes are used to provide a docstring for the function, which describes what the function does, its arguments, and its return value. This is useful for documentation and for tools that can automatically generate documentation from code.

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The weather in Hyderabad is sunny.


In [3]:
# Using multiple tools
from dotenv import load_dotenv
import os
from smolagents import tool
import requests
from smolagents import CodeAgent, LiteLLMModel, tool

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key is not None)

model = LiteLLMModel(
    model_id="gemini/gemini-3.6-flash",
    api_key=api_key
)
@tool
def get_weather(city: str) -> str:
    """
    Gets the current weather for a city.

    Args:
        city: The name of the city to get the current weather for.
    """

    # Find the coordinates of the city
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
    )

    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"Could not find the city {city}."

    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    # Get current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,weather_code",
            "timezone": "auto"
        }
    )

    weather_data = weather_response.json()

    current = weather_data["current"]

    return (
        f"City: {city}\n"
        f"Temperature: {current['temperature_2m']}°C\n"
        f"Humidity: {current['relative_humidity_2m']}%\n"
        f"Weather code: {current['weather_code']}"
    )
@tool
def calculator(expression: str) -> str:
    """
    Calculates a mathematical expression.

    Args:
        expression: The mathematical expression to calculate.
    """
    try:
        return str(eval(expression))
    except Exception:
        return "Could not calculate the expression."
@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """
    Converts a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.
    """
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{fahrenheit:.1f}°F"
agent = CodeAgent(
    tools=[get_weather, calculator, celsius_to_fahrenheit],
    model=model
)

True


In [4]:
agent_response = agent.run(
    "Get the weather in Hyderabad, calculate 2 + 2, and convert 25°C to Fahrenheit."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Get the weather in Hyderabad, calculate 2 + 2, and convert 25°C to Fahrenheit.                                  │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.6-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather_hyderabad = get_weather(city="Hyderabad")                                                                
  calc_result = calculator(expression="2 + 2")                                                                     
  temp_fahrenheit = celsius_to_fahrenheit(celsius=25)                                                              
                                                                                                                   
  print(f"Weather in Hyderabad: {weather_hyderabad}")                                                              
  print(f"2 + 2 = {calc_result}")                                                                                  
  print(f"25°C in Fahrenheit: {temp_fahrenheit}")                                                                  
                                                                                                                   
  result = {                                                                                                       
      "weather_in_hyderabad": weather_hyderabad,                                                                   
      "calculator_result": calc_result,                                                                            
      "celsius_25_in_fahrenheit": temp_fahrenheit                                                                  
  }                                                                                                                
                                                                                                                   
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Weather in Hyderabad: City: Hyderabad
Temperature: 33.4°C
Humidity: 36%
Weather code: 51
2 + 2 = 4
25°C in Fahrenheit: 77.0°F

Code execution exceeded the maximum execution time of 30 seconds

...

[Step 3: Duration 27.05 seconds| Input tokens: 8,097 | Output tokens: 989]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Weather in Hyderabad: Temperature 33.4°C, Humidity 36%, Weather code 51. 2 + 2 = 4. 25°C is        
  77.0°F.")                                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Weather in Hyderabad: Temperature 33.4°C, Humidity 36%, Weather code 51. 2 + 2 = 4. 25°C is 77.0°F.

[Step 4: Duration 4.61 seconds| Input tokens: 11,322 | Output tokens: 1,124]

Rag Implementation 


In [5]:

import re
from pypdf import PdfReader

pdf_path = "villa marie degree college for women HAND-BOOK-2024-2027.pdf"

reader = PdfReader(pdf_path)


def clean(text: str) -> str:
    # Fix words split across lines, such as attend-\nance
    text = re.sub(r"-\n(\w)", r"\1", text)

    # Join single line breaks inside paragraphs
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Remove extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


# Extract and clean text page by page
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = clean(page.extract_text() or "")

    pages.append({
        "page": page_number,
        "text": text
    })


# Check for pages with little or no extracted text
empty = [
    p["page"] for p in pages
    if len(p["text"]) < 50
]

print("Total PDF pages:", len(pages))
print("Pages with little/no text:", empty)

# Combine all cleaned page text into one string
document = "\n\n".join(
    f"--- Page {p['page']} ---\n{p['text']}"
    for p in pages
)

print("Total characters:", len(document))
print("\nFirst 2000 characters:\n")
print(document[:2000])

Total PDF pages: 70
Pages with little/no text: [4]
Total characters: 95122

First 2000 characters:

--- Page 1 ---
1 Villa Marie Degree College for Women Somajiguda, Hyderabad - 500082 Affiliated to Osmania University, Management Programs Approved by AICTE Recognized by UGC u/s 2(f), an ISO 9001:2015 Certified Institution Accredited with B++ Grade by NAAC College Code: 1238 STUDENT HAND BOOK 2024-2027

--- Page 2 ---
2 GENERAL INFORMATION ADDRESS: Villa Marie Degree College for Women 6-3-1089, Raj Bhavan Road Somajiguda Hyderabad. 500082 COMMUNICATION: Phone: +91-40-23391966, 2330 6302 Mail: villamarieedu@gmail.com Website: www.villamariecollege.ac.in NETWORK @villacollegepress @villamariecollege @villaedunexus

--- Page 3 ---
3 STUDENTPROFILE Name: ______________________ Class: ________________ Section: _________________________ Program: ______________________ Academic Year: ______________________ Hall Ticket No.:______________________ E-Mail ID: ______________________ Phone No.: ____

In [6]:

def create_chunks(text, chunk_size=1000, overlap=150):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Invalid chunk_size or overlap.")

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append(chunk_text)

        if end == len(text):
            break

        start = end - overlap

    return chunks


# Create chunks from the extracted PDF text
chunks = create_chunks(document)

# Check the results
print("\nTotal chunks:", len(chunks))

for i in range(min(3, len(chunks))):
    print(f"\n--- Chunk {i} ---")
    print(chunks[i])
    print("-" * 50)


Total chunks: 112

--- Chunk 0 ---
--- Page 1 ---
1 Villa Marie Degree College for Women Somajiguda, Hyderabad - 500082 Affiliated to Osmania University, Management Programs Approved by AICTE Recognized by UGC u/s 2(f), an ISO 9001:2015 Certified Institution Accredited with B++ Grade by NAAC College Code: 1238 STUDENT HAND BOOK 2024-2027

--- Page 2 ---
2 GENERAL INFORMATION ADDRESS: Villa Marie Degree College for Women 6-3-1089, Raj Bhavan Road Somajiguda Hyderabad. 500082 COMMUNICATION: Phone: +91-40-23391966, 2330 6302 Mail: villamarieedu@gmail.com Website: www.villamariecollege.ac.in NETWORK @villacollegepress @villamariecollege @villaedunexus

--- Page 3 ---
3 STUDENTPROFILE Name: ______________________ Class: ________________ Section: _________________________ Program: ______________________ Academic Year: ______________________ Hall Ticket No.:______________________ E-Mail ID: ______________________ Phone No.: ______________________ Address: ____________________________________

In [7]:

from sentence_transformers import SentenceTransformer

# Model used only for embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for the handbook chunks
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 4/4 [01:49<00:00, 27.40s/it]

Embedding shape: (112, 384)


In [8]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def retrieve_chunks(question, top_k=4):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "chunk": chunks[index],
            "score": float(scores[index]),
            "index": int(index)
        })

    return results


question = "What is the attendance requirement?"
results = retrieve_chunks(question)

for result in results:
    print(f"\nChunk {result['index']}")
    print(f"Similarity: {result['score']:.3f}")
    print(result["chunk"])
    print("-" * 50)


Chunk 22
Similarity: 0.666
rmediate Education  Candidates must pay the First Semester Fees at the Acco unts Section in the Office to confirm and complete the admission in the chosen program. CODE OF CONDUCT  Students should be punctual and attend all their classes regularly.  Students must be decently dressed and be presentable to the college.  Students should wear their College Identity Cards in the College premises.

--- Page 18 ---
18  The college lays emphasis on attendance. It is mandatory for the students to have a minimum of 75% attendance in each subject to be eligible to appear for Continuous Internal Evaluation (CIE) i.e., Unit Tests, Internal Assessment Tests, Semester -End Examinations, etc.  The list of students having shortage of attendance is displayed on the respective Departmental Notice Boards every month. The same is also communicated to the Parents.  Students representing the College, University, State or Nation in Sports, NSS, NCC or Cultural activities sha

In [9]:
# Using multiple tools
from dotenv import load_dotenv
import os
from smolagents import tool
import requests
from smolagents import CodeAgent, LiteLLMModel, tool

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key is not None)

model = LiteLLMModel(
    model_id="gemini/gemini-3.6-flash",
    api_key=api_key
)
@tool
def get_weather(city: str) -> str:
    """
    Gets the current weather for a city.

    Args:
        city: The name of the city to get the current weather for.
    """

    # Find the coordinates of the city
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
    )

    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"Could not find the city {city}."

    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    # Get current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,weather_code",
            "timezone": "auto"
        }
    )

    weather_data = weather_response.json()

    current = weather_data["current"]

    return (
        f"City: {city}\n"
        f"Temperature: {current['temperature_2m']}°C\n"
        f"Humidity: {current['relative_humidity_2m']}%\n"
        f"Weather code: {current['weather_code']}"
    )
@tool
def calculator(expression: str) -> str:
    """
    Calculates a mathematical expression.

    Args:
        expression: The mathematical expression to calculate.
    """
    try:
        return str(eval(expression))
    except Exception:
        return "Could not calculate the expression."
@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """
    Converts a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.
    """
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{fahrenheit:.1f}°F"
@tool
def search_college_docs(question: str) -> str:
    """
    Search the Villa Marie Degree College handbook for information
    about attendance, college rules, backlogs, examinations,
    courses, and the B.Sc. Data Science syllabus.

    Args:
        question: The question to search for in the college handbook.

    Returns:
        The most relevant handbook passages and their similarity scores.
    """
    results = retrieve_chunks(question, top_k=2)

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Chunk {result['index']} "
            f"(similarity: {result['score']:.3f}):\n"
            f"{result['chunk']}"
        )

    return "\n\n---\n\n".join(formatted_results)
agent = CodeAgent(
    tools=[get_weather, calculator, celsius_to_fahrenheit, search_college_docs],
    model=model,
    max_steps= 3
)

True


In [10]:
agent.run("what is the attendance requirement?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ what is the attendance requirement?                                                                             │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.6-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  attendance_info = search_college_docs(question="attendance requirement")                                         
  print(attendance_info)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Chunk 22 (similarity: 0.684):
rmediate Education  Candidates must pay the First Semester Fees at the Acco unts Section in the Office to confirm 
and complete the admission in the chosen program. CODE OF CONDUCT  Students should be punctual and attend all 
their classes regularly.  Students must be decently dressed and be presentable to the college.  Students should 
wear their College Identity Cards in the College premises.

--- Page 18 ---
18  The college lays emphasis on attendance. It is mandatory for the students to have a minimum of 75% attendance 
in each subject to be eligible to appear for Continuous Internal Evaluation (CIE) i.e., Unit Tests, Internal 
Assessment Tests, Semester -End Examinations, etc.  The list of students having shortage of attendance is 
displayed on the respective Departmental Notice Boards every month. The same is also communicated to the Parents. 
Students representing the College, University, State or Nation in Sports, NSS, NCC or Cultural activities shall be 
eligible to claim the

---

Chunk 23 (similarity: 0.574):
e Parents.  Students representing the College, University, State or Nation in Sports, NSS, NCC or Cultural 
activities shall be eligible to claim the attendance for the actual number of days participated , subject to a 
maximum of 20 days in a semester based on the specific recommendations of the HOD and the Principal of the College.
 Students are required to maintain discipline which is essential for the progress and development of the 
individual and the college.  Students must conduct themselves in a manner befitting the college comm unity with 
due respect given to all the staff members.  No candidate shall create mutual hatred or cause tension between 
different castes and communities, religious or linguistic or between any group(s) of students.  No candidate shall
be permitted to carry procession or public meeting.  No candidate shall, nor shall her supporters, deface or cause
any destruction to any property of the college.  All candidates shall be prohibited from indulging or

Out: None

[Step 1: Duration 9.90 seconds| Input tokens: 2,358 | Output tokens: 132]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The attendance requirement is a minimum of 75% attendance in each subject to be eligible to        
  appear for Continuous Internal Evaluation (CIE), including Unit Tests, Internal Assessment Tests, and            
  Semester-End Examinations. Additionally, students representing the college in sports, NSS, NCC, or cultural      
  activities can claim attendance for the actual number of days participated (up to a maximum of 20 days in a      
  semester) subject to recommendation.")                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The attendance requirement is a minimum of 75% attendance in each subject to be eligible to appear 
for Continuous Internal Evaluation (CIE), including Unit Tests, Internal Assessment Tests, and Semester-End 
Examinations. Additionally, students representing the college in sports, NSS, NCC, or cultural activities can claim
attendance for the actual number of days participated (up to a maximum of 20 days in a semester) subject to 
recommendation.

[Step 3: Duration 2.88 seconds| Input tokens: 8,834 | Output tokens: 457]

'The attendance requirement is a minimum of 75% attendance in each subject to be eligible to appear for Continuous Internal Evaluation (CIE), including Unit Tests, Internal Assessment Tests, and Semester-End Examinations. Additionally, students representing the college in sports, NSS, NCC, or cultural activities can claim attendance for the actual number of days participated (up to a maximum of 20 days in a semester) subject to recommendation.'